# Contagem de inversões em um vetor

**Definição.** Em um vetor `A`, há uma inversão quando existem índices `i < j` tais que `A[i] > A[j]`.

Exemplo:

```python
A = [4, 1, 3, 2]
```

Inversões:

- `(4, 1)`
- `(4, 3)`
- `(4, 2)`
- `(3, 2)`

Total: `4` inversões.

A solução eficiente usa a lógica do **MergeSort**.

In [ ]:
%load_ext cython

## Por que usar MergeSort?

A contagem ingênua testa todos os pares `(i, j)` e custa **Θ(n²)**.

Com MergeSort, contamos inversões durante a intercalação. Quando um elemento da metade direita é menor que um elemento da metade esquerda, ele também é menor que todos os elementos restantes da metade esquerda, pois essa metade já está ordenada.

Se `L[i] > R[j]`, então o número de novas inversões é:

\[
|L| - i
\]

### Análise assintótica

A recorrência é a mesma do MergeSort:

\[
T(n) = 2T(n/2) + \Theta(n)
\]

Portanto:

\[
T(n) = \Theta(n \log n)
\]

O espaço auxiliar é **Θ(n)**.

In [ ]:
%%cython
# cython: boundscheck=False, wraparound=False, cdivision=True

cdef long _merge_conta(list A, Py_ssize_t esquerda, Py_ssize_t meio, Py_ssize_t direita):
    cdef list L = A[esquerda:meio + 1]
    cdef list R = A[meio + 1:direita + 1]
    cdef Py_ssize_t i = 0, j = 0, k = esquerda
    cdef Py_ssize_t nL = len(L), nR = len(R)
    cdef long inversoes = 0

    while i < nL and j < nR:
        if <long>L[i] <= <long>R[j]:
            A[k] = L[i]
            i += 1
        else:
            A[k] = R[j]
            inversoes += nL - i
            j += 1
        k += 1

    while i < nL:
        A[k] = L[i]
        i += 1
        k += 1

    while j < nR:
        A[k] = R[j]
        j += 1
        k += 1

    return inversoes

cdef long _merge_sort_conta(list A, Py_ssize_t esquerda, Py_ssize_t direita):
    cdef Py_ssize_t meio
    cdef long inv = 0

    if esquerda >= direita:
        return 0

    meio = (esquerda + direita) // 2
    inv += _merge_sort_conta(A, esquerda, meio)
    inv += _merge_sort_conta(A, meio + 1, direita)
    inv += _merge_conta(A, esquerda, meio, direita)
    return inv

def contar_inversoes(list A):
    cdef list copia = list(A)
    cdef long inv = _merge_sort_conta(copia, 0, len(copia) - 1)
    return inv, copia

In [ ]:
A = [4, 1, 3, 2]
inv, ordenado = contar_inversoes(A)
print("Vetor original:", A)
print("Vetor ordenado:", ordenado)
print("Inversões:", inv)

## Linha crítica da otimização

A linha conceitual mais importante é:

```python
inversoes += nL - i
```

Ela evita contar uma por uma as inversões entre `R[j]` e os elementos restantes da esquerda. Como a metade esquerda já está ordenada, todos os elementos `L[i], L[i+1], ..., L[nL-1]` são maiores que `R[j]`.

Essa é a melhoria que reduz a contagem de **Θ(n²)** para **Θ(n log n)**.

In [ ]:
testes = [
    [1, 2, 3, 4],
    [4, 3, 2, 1],
    [2, 1, 3, 1],
    [],
]

for t in testes:
    print(t, "=>", contar_inversoes(t))